<a href="https://colab.research.google.com/github/baanujan-18/Statistical-Learning-e20030/blob/main/Assignment_07_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q1. Bayesian Estimation of a User Ability Parameter from Item Responses

## 1. Visualizing the Mechanics of the 2PL Model

The two-parameter logistic item-response probability is

$$
p_i(\theta)
=
P(Y_i=1\mid\Theta=\theta)
=
\frac{1}{1+\exp[-a_i(\theta-b_i)]}.
$$

Here, $a_i>0$ is the discrimination parameter and $b_i$ is the difficulty parameter.

The difficulty parameter $b_i$ controls the horizontal position of the curve.

- Increasing $b_i$ shifts the curve to the right.
- Decreasing $b_i$ shifts the curve to the left.
- At $\theta=b_i$,

$$
p_i(b_i)=\frac{1}{2}.
$$

Therefore, a larger value of $b_i$ means that a user requires a larger ability value to obtain the same probability of answering correctly.

The discrimination parameter $a_i$ controls the steepness of the curve. A large value of $a_i$ gives a steep curve, while a small value of $a_i$ gives a flatter curve.

In [1]:
import numpy as np
import plotly.graph_objects as go

def p_2pl(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_values = np.linspace(-6, 6, 500)

configurations = [
    {"a": 0.5, "b": 0, "dash": "dash"},
    {"a": 1.5, "b": -2, "dash": "solid"},
    {"a": 1.5, "b": 0, "dash": "solid"},
    {"a": 1.5, "b": 2, "dash": "solid"}
]

fig = go.Figure()

for config in configurations:
    probabilities = p_2pl(
        theta_values,
        config["a"],
        config["b"]
    )

    fig.add_trace(
        go.Scatter(
            x=theta_values,
            y=probabilities,
            mode="lines",
            name=f"a = {config['a']}, b = {config['b']}",
            line=dict(
                dash=config["dash"],
                width=2.5
            )
        )
    )

fig.update_layout(
    title="Two-Parameter Logistic Item Response Curves",
    xaxis_title="Latent Ability, θ",
    yaxis_title="P(Yᵢ = 1 | Θ = θ)",
    xaxis=dict(range=[-6, 6]),
    yaxis=dict(range=[0, 1.02]),
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

## 2. Sequential Likelihood Contribution

At step $k$, the observed response is $y_k\in\{0,1\}$.

The likelihood contribution of one response is

$$
L(y_k\mid\theta)
=
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k},
$$

where

$$
p_k(\theta)
=
\frac{1}{1+\exp[-a_k(\theta-b_k)]}.
$$

If the response is correct, $y_k=1$, then

$$
L(y_k\mid\theta)=p_k(\theta).
$$

If the response is incorrect, $y_k=0$, then

$$
L(y_k\mid\theta)=1-p_k(\theta).
$$

Let the running history be

$$
\mathbf y^{(k)}=(y_1,y_2,\ldots,y_k).
$$

Assuming that the item responses are conditionally independent given $\Theta=\theta$, the joint likelihood is

$$
L(\mathbf y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
[p_i(\theta)]^{y_i}
[1-p_i(\theta)]^{1-y_i}.
$$

## 3. Mathematical Formulation of the Running Update

Let

$$
f_{k-1}(\theta)
=
f_{\Theta\mid\mathbf Y^{(k-1)}}
\left(
\theta\mid\mathbf y^{(k-1)}
\right)
$$

be the posterior density after the first $k-1$ responses. This posterior becomes the prior density for step $k$.

After observing the new response $y_k$, Bayes' theorem gives

$$
f_k(\theta)
=
\frac{
L(y_k\mid\theta)f_{k-1}(\theta)
}{
\displaystyle
\int_{-\infty}^{\infty}
L(y_k\mid s)f_{k-1}(s)\,ds
}.
$$

Therefore, up to a proportionality constant,

$$
f_k(\theta)
\propto
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{k-1}(\theta).
$$

Before any responses are observed, the initial prior is the standard normal density

$$
f_0(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right).
$$

## 4. Dynamic Shifting After a Correct Difficult Item

Suppose that the user answers a highly difficult item correctly. Then

$$
y_k=1
$$

and $b_k$ is large.

The posterior update becomes

$$
f_k(\theta)
\propto
p_k(\theta)f_{k-1}(\theta).
$$

For a difficult item, $p_k(\theta)$ is small for low values of $\theta$ and becomes large only when $\theta$ approaches or exceeds $b_k$.

Multiplication by $p_k(\theta)$ therefore:

- reduces posterior density at low ability values;
- keeps relatively more posterior density at high ability values;
- shifts the posterior mean toward larger values of $\theta$;
- shifts the posterior mode toward larger values of $\theta$.

Using the log-posterior,

$$
\log f_k(\theta)
=
\log f_{k-1}(\theta)
+
\log p_k(\theta)
+
C,
$$

where $C$ does not depend on $\theta$.

For a correct response,

$$
\frac{d}{d\theta}\log p_k(\theta)
=
a_k[1-p_k(\theta)]>0.
$$

Thus, the new likelihood adds a positive slope to the previous log-posterior and shifts its peak to the right.

## 5. Tracking Certainty and Sharpness

The derivative of the item response probability is

$$
p_k'(\theta)
=
a_kp_k(\theta)[1-p_k(\theta)].
$$

The information supplied by item $k$ is

$$
I_k(\theta)
=
a_k^2p_k(\theta)[1-p_k(\theta)].
$$

Therefore, the information supplied by an item increases with $a_k^2$.

When $a_k$ is very large:

- the item response curve is steep;
- the response strongly separates ability values below and above $b_k$;
- the likelihood contains more information;
- the posterior becomes sharper;
- the posterior variance generally decreases more strongly.

When $a_k$ is very small:

- the item response curve is flat;
- the response probability changes slowly with ability;
- the item supplies little information;
- the posterior changes only slightly;
- the posterior variance decreases very little.

The maximum item information occurs near $\theta=b_k$, because

$$
p_k(b_k)=0.5.
$$

Therefore,

$$
I_k(b_k)=\frac{a_k^2}{4}.
$$

## 6. Numerical Implementation Using a Fixed Grid

Because the normal prior is not conjugate to the 2PL likelihood, the posterior does not have a standard closed-form distribution. It can be approximated numerically on a fixed grid.

First, define grid points

$$
\theta_1,\theta_2,\ldots,\theta_m
$$

over a sufficiently wide interval, such as $[-5,5]$.

Evaluate the initial standard normal prior at each grid point:

$$
f_0(\theta_j)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta_j^2}{2}\right).
$$

After response $y_k$ is observed, calculate

$$
p_k(\theta_j)
=
\frac{1}{1+\exp[-a_k(\theta_j-b_k)]}.
$$

The likelihood at each grid point is

$$
L_j
=
[p_k(\theta_j)]^{y_k}
[1-p_k(\theta_j)]^{1-y_k}.
$$

The unnormalized posterior is

$$
\widetilde f_k(\theta_j)
=
L_jf_{k-1}(\theta_j).
$$

The numerical normalizing constant is calculated with the trapezoidal rule:

$$
Z_k
\approx
\operatorname{trapz}
\left(
\widetilde f_k(\theta_j),\theta_j
\right).
$$

The normalized posterior is

$$
f_k(\theta_j)
=
\frac{\widetilde f_k(\theta_j)}{Z_k}.
$$

This normalized posterior becomes the prior for the next response.

The posterior mean is approximated by

$$
\widehat\theta_{\mathrm{Bayes}}^{(k)}
\approx
\operatorname{trapz}
\left(
\theta_jf_k(\theta_j),\theta_j
\right).
$$

The MAP estimate is

$$
\widehat\theta_{\mathrm{MAP}}^{(k)}
\approx
\theta_{\arg\max_j f_k(\theta_j)}.
$$

In [2]:
import numpy as np
from scipy.stats import norm

def p_2pl(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_grid = np.linspace(-5, 5, 1000)

posterior = norm.pdf(theta_grid, loc=0, scale=1)
posterior /= np.trapezoid(posterior, theta_grid)

def update_posterior(
    previous_posterior,
    theta_grid,
    response,
    discrimination,
    difficulty
):
    probability = p_2pl(
        theta_grid,
        discrimination,
        difficulty
    )

    likelihood = (
        probability ** response
        * (1 - probability) ** (1 - response)
    )

    unnormalized_posterior = (
        previous_posterior * likelihood
    )

    normalizing_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    if normalizing_constant <= 0:
        raise ValueError("Posterior normalization failed.")

    new_posterior = (
        unnormalized_posterior
        / normalizing_constant
    )

    posterior_mean = np.trapezoid(
        theta_grid * new_posterior,
        theta_grid
    )

    posterior_map = theta_grid[
        np.argmax(new_posterior)
    ]

    return new_posterior, posterior_mean, posterior_map

In [3]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import norm

# Random-number generator
rng = np.random.default_rng(42)

# True hidden ability
theta_true = 0.75

# Number of items
n_items = 20

# Ability grid
theta_grid = np.linspace(-5, 5, 1500)

def p_2pl(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Random item characteristics
a_values = rng.uniform(
    0.5,
    2.0,
    size=n_items
)

b_values = rng.normal(
    0.0,
    1.0,
    size=n_items
)

# Initial N(0,1) prior
posterior = norm.pdf(
    theta_grid,
    loc=0.0,
    scale=1.0
)

posterior /= np.trapezoid(
    posterior,
    theta_grid
)

# Estimates at step 0
posterior_means = [
    np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )
]

map_estimates = [
    theta_grid[np.argmax(posterior)]
]

responses = []
true_probabilities = []

for k in range(n_items):
    a_k = a_values[k]
    b_k = b_values[k]

    # True probability of a correct response
    p_true = p_2pl(
        theta_true,
        a_k,
        b_k
    )

    true_probabilities.append(p_true)

    # Simulate response
    uniform_draw = rng.uniform(0, 1)

    if uniform_draw < p_true:
        y_k = 1
    else:
        y_k = 0

    responses.append(y_k)

    # Likelihood across the grid
    p_grid = p_2pl(
        theta_grid,
        a_k,
        b_k
    )

    likelihood = (
        p_grid ** y_k
        * (1 - p_grid) ** (1 - y_k)
    )

    # Sequential update
    posterior = posterior * likelihood

    # Normalize posterior
    normalizing_constant = np.trapezoid(
        posterior,
        theta_grid
    )

    posterior = (
        posterior / normalizing_constant
    )

    # Posterior mean
    posterior_mean_k = np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )

    # MAP estimate
    map_k = theta_grid[
        np.argmax(posterior)
    ]

    posterior_means.append(
        posterior_mean_k
    )

    map_estimates.append(
        map_k
    )

steps = np.arange(n_items + 1)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text=(
        f"True ability θ = {theta_true}"
    ),
    annotation_position="top right"
)

fig.update_layout(
    title="Sequential Estimation of User Ability",
    xaxis_title="Number of Observed Items, k",
    yaxis_title="Estimated Ability, θ̂",
    xaxis=dict(
        tickmode="linear",
        tick0=0,
        dtick=1
    ),
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

results = pd.DataFrame({
    "Step": np.arange(1, n_items + 1),
    "Discrimination a_k": a_values,
    "Difficulty b_k": b_values,
    "True P(correct)": true_probabilities,
    "Response y_k": responses,
    "Posterior Mean": posterior_means[1:],
    "MAP Estimate": map_estimates[1:]
})

display(results.round(4))

,Step,Discrimination a_k,Difficulty b_k,True P(correct),Response y_k,Posterior Mean,MAP Estimate
0,1,1.6609,-0.1849,0.8253,1,0.5016,0.4370
1,2,1.1583,-0.6809,0.8399,1,0.6656,0.5837
2,3,1.7879,1.2225,0.3005,0,0.3916,0.3903
3,4,1.5461,-0.1545,0.8019,1,0.5869,0.5637
4,5,0.6413,-0.4283,0.6804,0,0.4312,0.4103
5,6,1.9634,-0.3521,0.8970,1,0.5637,0.5170
6,7,1.6417,0.5323,0.5884,1,0.7771,0.7305
7,8,1.6791,0.3654,0.6560,1,0.9200,0.8706
8,9,0.6922,0.4127,0.5581,0,0.8200,0.7772
9,10,1.1756,0.4308,0.5927,1,0.9225,0.8773


### Convergence Analysis

At the beginning of the sequence, the posterior mean and MAP estimate may fluctuate substantially because only a small number of responses have been observed.

As the number of observed items $k$ increases:

- the posterior mean and MAP estimate generally move closer to the true ability $\theta_{\mathrm{true}}=0.75$;
- the influence of the initial prior becomes smaller;
- the influence of a single unusual response becomes smaller;
- the posterior distribution becomes narrower;
- the estimates become more stable.

The convergence does not need to be monotonic because the responses are random. A correct or incorrect response can temporarily move the estimates away from the true value.

Items with larger discrimination values normally produce larger updates because they contain more information.

A decreasing distance between the estimates and $\theta_{\mathrm{true}}$, together with a narrowing posterior distribution, indicates that the platform is becoming more confident about the user's ability.

# Q2. Bayesian Tracking of CTR via Beta-Binomial Updates